In [1]:
import pandas as pd
import csv
import pandas_gbq
import time
from datetime import datetime
import os
from google.oauth2 import service_account
from google.cloud import bigquery

In [2]:
CREDS = '../converge-database-0331482f2ee5.json'

In [ ]:
client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)

In [ ]:
seriatim = pd.DataFrame()
premium = pd.DataFrame()
withdrawals = pd.DataFrame()
terminated = pd.DataFrame()
annuitized = pd.DataFrame()

In [ ]:
file="I:/New Structure/Actuarial New/Database/KSKJ/202604/KSKJ Converge Report 20260430.xlsx"

In [ ]:
seriatim = pd.read_excel(file, dtype="str", sheet_name = "Seriatim")

In [ ]:
premium = pd.read_excel(file, dtype="str", sheet_name = "Premiums")

In [ ]:
withdrawals = pd.read_excel(file, dtype="str", sheet_name = "Withdrawals")

In [ ]:
terminated = pd.read_excel(file, dtype="str", sheet_name = "Terminated Policies")

In [ ]:
annuitized = pd.read_excel(file, dtype="str", sheet_name = "Annuitized Policies")

In [ ]:
seriatim.info()

In [ ]:
# Lowercasing the headers and removing the spaces between them
seriatim.columns = seriatim.columns.str.strip()
seriatim.columns = seriatim.columns.map(str.lower)
seriatim.columns = seriatim.columns.map(lambda x : x.replace(" ", "_"))
seriatim.columns = seriatim.columns.map(lambda x : x.replace("/", "_"))
seriatim.columns = seriatim.columns.map(lambda x : x.replace("+", "_plus"))

In [ ]:
seriatim = seriatim.rename(columns={"additional_premiums.1": "additional_premiums_mtd", "inforce_date" : "set_month", "additional_premiums_total" : "additional_premiums_mtd", "stat_reserves": "stat_reserve", "tax_reserves":"tax_reserve" })

In [ ]:
seriatim.set_month=seriatim.set_month.astype("datetime64[ns]")
seriatim['set_month']= seriatim['set_month'].dt.strftime('%Y%m')
set_month = seriatim.set_month[0]

In [ ]:
# automatically deleting unwant rows
valid_rows = seriatim.iloc[:, 0].dropna().shape[0]
seriatim = seriatim.iloc[:valid_rows, :]

seriatim = seriatim.loc[:, ~seriatim.columns.str.contains("unnamed")]
seriatim

In [ ]:
seriatim = seriatim.astype({"issue_date":"datetime64[ns]","renewal_date":"datetime64[ns]", "guarantee_period_end_date" :"datetime64[ns]","maturity_date":"datetime64[ns]", "date_approved" :"datetime64[ns]","issue_age" :"int64",
                     "purchase_price" :"float64","additional_premiums" : "float64","total_premiums" :"float64", "bom_fund_value" : "float64", "interest_credited" : "float64",
                     "bonus_credited" : "float64", "additional_premiums_mtd" : "float64", "rmd_withdrawals" : "float64", "free_interest_credit_withdrawals" : "float64",
                     "freelook_withdrawals" : "float64","cancellation_withdrawals" :"float64","death_benefit" : "float64",
                     "enhanced_benefit_withdrawals" : "float64", "free_partial_withdrawals" : "float64", "partial_withdrawal_with_sc" : "float64",
                      "full_surrender_withdrawals" : "float64", "surrender_charges" : "float64", "expense_charges" : "float64", "eom_fund_value" : "float64",
                     "cumulative_interest_credited" : "float64","cumulative_bonus_credited" :"float64","cumulative_additional_premiums" : "float64",
                     "cumulative_rmd_withdrawals" : "float64", "cumulative_free_interest_credit_withdrawals" : "float64", "cumulative_freelook_withdrawals" : "float64",
                     "cumulative_cancellation_withdrawals" : "float64", "cumulative_death_benefit" : "float64", "cumulative_enhanced_benefit_withdrawals" : "float64", "cumulative_free_partial_withdrawals" : "float64",
                     "cumulative_partial_withdrawal_with_sc" : "float64","cumulative_full_surrender_withdrawals" :"float64","cumulative_surrender_charges" : "float64",
                     "cumulative_expense_charges" : "float64", "gmsv" : "float64", "free_partial_withdrawal_rider" : "int64", "death_benefit_rider" : "int64",
                     "enhanced_benefit_rider" : "int64", "bonus_crediting_rider" : "int64", "year_1_int_rate" : "float64", "year_2_plus_int_rate" : "float64",
                     "guaranteed_minimum_crediting_rate" : "float64", "snfl_crediting_rate" : "float64", "renewal_indicator" : "int64", "surrender_value_w_o_mva" : "float64",
                     "surrender_value_w_mva" : "float64", "stat_reserve" : "float64", "tax_reserve" : "float64", "quota_share" : "float64", "set_month" : "object", "full_surr_w_sc" : "float64", "full_surr_wo_sc" : "float64",
                     "partial_surr_w_sc" : "float64", "partial_surr_wo_sc" : "float64", "cumulative_internal_reissues_withdrawals" : "float64", "account_value_at_renewal" : "float64", "internal_reissues_withdrawals" :"float64"
               })

In [ ]:
seriatim.to_gbq("converge-database.kskj.seriatim",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

In [ ]:
premium.info()

In [ ]:
#Lowercasing the headers and removing the spaces between them
premium.columns = premium.columns.str.strip()
premium.columns = premium.columns.map(str.lower)
premium.columns = premium.columns.map(lambda x : x.replace(" ", "_"))
premium.columns = premium.columns.map(lambda x : x.replace("/", "_"))

In [ ]:
premium = premium.astype({"issue_date" : "datetime64[ns]", "transaction_date" : "datetime64[ns]", "bom_fund_value" : "float64", "initial_premium" : "float64", 
                          "additional_premium" : "float64", "total_premium" : "float64", "renewal_premium" : "float64", "renewal_cede" : "float64",
                         "initial_cede": "float64", "initial_commission" : "float64", "renewal_commission" : "float64", "quota_share" :"float64"})

In [ ]:
premium = premium.iloc[:,0:15]

In [ ]:
premium['set_month'] = set_month

In [ ]:
premium

In [ ]:
premium.to_gbq("converge-database.kskj.premium",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

In [ ]:
withdrawals

In [ ]:
withdrawals.columns = withdrawals.columns.str.strip()
withdrawals.columns = withdrawals.columns.map(str.lower)
withdrawals.columns = withdrawals.columns.map(lambda x : x.replace(" ", "_"))
withdrawals.columns = withdrawals.columns.map(lambda x : x.replace("/", "_"))

In [ ]:
withdrawals

In [ ]:
withdrawals = withdrawals.dropna(subset =['policy_number'], axis=0)
withdrawals = withdrawals.drop(['comm_chargebacks'],axis=1)

In [ ]:
#withdrawals = withdrawals.iloc[:,0:12]

In [ ]:
# only delete unname column, keep common chargebacks
withdrawals = withdrawals.loc[:, ~withdrawals.columns.str.contains("^unnamed")]

In [ ]:
withdrawals = withdrawals.astype({"issue_date" : "datetime64[ns]", "transaction_date" : "datetime64[ns]", "bom_fund_value" : "float64", 
                          "withdrawal_amount" : "float64", "surrender_charge" : "float64", "eom_fund_value" : "float64", "quota_share" :"float64"})

In [ ]:
withdrawals['set_month'] = set_month

In [ ]:
withdrawals.info()

In [ ]:
withdrawals.to_gbq("converge-database.kskj.withdrawals",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

In [ ]:
terminated

In [ ]:
terminated.columns = terminated.columns.str.strip()
terminated.columns = terminated.columns.map(str.lower)
terminated.columns = terminated.columns.map(lambda x : x.replace(" ", "_"))
terminated.columns = terminated.columns.map(lambda x : x.replace("/", "_"))

In [ ]:
terminated = terminated.astype({"issue_date" : "datetime64[ns]", "termination_date" : "datetime64[ns]", "cumulative_premium" : "float64", 
                          "cumulative_interest_earned" : "float64", "withdrawal_amount" : "float64", "cumulative_surrender_charge" : "float64", "fund_value_on_termination" :"float64", 'quota_share' : 'float64'})

In [ ]:
terminated['set_month'] = set_month

In [ ]:
terminated.info()


In [ ]:
terminated.to_gbq("converge-database.kskj.terminated",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

In [ ]:
annuitized

In [ ]:
annuitized.columns = annuitized.columns.str.strip()
annuitized.columns = annuitized.columns.map(str.lower)
annuitized.columns = annuitized.columns.map(lambda x : x.replace(" ", "_"))
annuitized.columns = annuitized.columns.map(lambda x : x.replace("/", "_"))

In [ ]:
annuitized = annuitized.rename(columns={"current_annuitized_account_value" : "annuitized_account_value"})

In [ ]:
annuitized = annuitized.astype({"issue_date" : "datetime64[ns]", "transaction_date" : "datetime64[ns]", "annuitized_account_value" : "float64", 
                          "annuitized_payment" : "float64", "annuitization_method" : "object", "annuitization_interest" : "float64", "annuitized_reserves" :"float64", 'quota_share' : 'float64'})

In [ ]:
annuitized= annuitized.iloc[:, 0:13]

In [ ]:
annuitized = annuitized.dropna(subset =['policy_number'] ,axis=0)

In [ ]:
annuitized = annuitized.drop({"annuitization_mortality_table"}, axis=1)

In [ ]:
annuitized['set_month'] = set_month

In [ ]:
annuitized.info()

In [ ]:
annuitized.to_gbq("converge-database.kskj.annuitized",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

In [ ]:
reported_date_q = f'''UPDATE `kskj.seriatim`  s
SET s.reported_date = (SELECT MIN(s2.set_month) FROM `kskj.seriatim` s2 WHERE s2.policy_number = s.policy_number)
WHERE s.reported_date is NULL'''

job = client.query(reported_date_q) 

In [ ]:
# query_to_fix ='UPDATE `kskj.seriatim`\
# SET plangroup = CONCAT(LEFT(plangroup,3), "E", right(plangroup,2))\
# WHERE set_month ='202409'\

In [ ]:
sys.path.append('../actuarial-pipelines/reconciliations/kskj')

from reconciliation import run_reconciliation

run_reconciliation(set_month)

In [ ]:
# Add directory containing LDTI.py
sys.path.append('../actuarial-pipelines/ldti')

# Import the function
from LDTI import main_query_run

# Trigger the AVRF analysis
main_query_run("KSKJ")


In [2]:
set_month ='202604'

In [3]:
import sys

# Add directory containing LDTI.py
sys.path.append('../actuarial-pipelines/avrf/kskj/')

# Import the function
from avrf import run_avrf_analysis

# Trigger the AVRF analysis
run_avrf_analysis(set_month)

AVRF analysis starting for kskj set_month 202604
Total absolute difference in AVRF AV: 6,157.52
Results saved to Query Results/AVRF/AVRF_kskj_202604.xlsx


,policy_number,issue_date,beginning_fund_value,premium,interest_credited,bonus_credited,rmd_withdrawals,free_interest_credit_withdrawals,freelook_withdrawals,cancellation_withdrawals,...,beginning_reserve_stat,end_reserve_stat,new_policy_check,dropped_policy_check,plan,plangroup,inflow,outflow,exp_av,diff
0,KJ30101001P1,2025-09-10,317443.5070,0.0000,1360.5615,0.0,0.0,0.0,0.0,0.0,...,317276.4400,318641.6850,0,0,MYGE24,MYG05,1360.5615,0.0,318804.0685,0.000000e+00
1,KJ30101002P1,2025-09-08,127271.9275,0.0000,545.4900,0.0,0.0,0.0,0.0,0.0,...,127205.5700,127753.0550,0,0,MYGE24,MYG05,545.4900,0.0,127817.4175,-2.910383e-11
2,KJ30101003P1,2025-08-26,31999.7145,0.0000,140.8280,0.0,0.0,0.0,0.0,0.0,...,33344.1355,33465.0990,0,0,MYGE24-IRA,MYG05,140.8280,0.0,32140.5425,-3.637979e-12
3,KJ30101004P1,2025-08-20,109024.7645,0.0000,467.2765,0.0,0.0,0.0,0.0,0.0,...,108995.3050,109463.5600,0,0,MYGE24-IRA,MYG05,467.2765,0.0,109492.0410,-1.455192e-11
4,KJ30101005P1,2025-08-26,14824.1135,0.0000,64.1060,0.0,0.0,0.0,0.0,0.0,...,15416.3150,15472.0230,0,0,MYGE24-IRA,MYG05,64.1060,0.0,14888.2195,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
740,KJ70104130P1,2026-04-21,0.0000,51216.6185,65.2650,0.0,0.0,0.0,0.0,0.0,...,0.0000,51254.4095,1,0,MYGE24-IRA,MYG08,51281.8835,0.0,51281.8835,-7.275958e-12
741,KJ70104131P1,2026-04-21,0.0000,27220.6160,35.6440,0.0,0.0,0.0,0.0,0.0,...,0.0000,27237.3930,1,0,MYGE24-ROT,MYG10,27256.2600,0.0,27256.2600,0.000000e+00
742,KJ70104132P1,2026-04-21,0.0000,52577.4745,66.9940,0.0,0.0,0.0,0.0,0.0,...,0.0000,52623.0840,1,0,MYGE24-IRA,MYG08,52644.4685,0.0,52644.4685,7.275958e-12
743,KJ70104133P1,2026-04-17,0.0000,86301.6195,153.0260,0.0,0.0,0.0,0.0,0.0,...,0.0000,86226.5885,1,0,MYGE24,MYG05,86454.6455,0.0,86454.6455,0.000000e+00
